#### BrainAI Sorter · BrainAI & BrainAI Coach Network
상업적 사용은 불가하며, 교육기관·학교의 학생 교육활동에는 자유롭게 사용할 수 있어요.

# 🧠 Step 4 - 모델 훈련 (Google Colab · 선택)

> **기본은 `Step 4 - Modeling (PC).ipynb` 예요.** PC 가 너무 느릴 때만 이 Colab 노트북을 쓰세요.
> **이 노트북은 Google Colab 에서 실행해요.** 사이트의 **"Colab 에서 열기"** 버튼으로 열면 돼요.
> 훈련 방법·전처리는 PC 노트북과 똑같아요.

**오늘 배우는 것**
- 🔁 **전이학습**: 미리 학습된 AI 모델(MobileNetV3 Large — ImageNet 100만 장 이상으로 학습)을 **내 새 데이터로 훈련해서** 쓰는 것. 우리는 앞부분(특징 뽑기)은 고정(얼림)하고, 새로 붙인 '내 색 판단' 부분만 훈련해요 (앞부분까지 함께 조금씩 훈련하는 '미세 조정'은 사진이 아주 많을 때)
- 📈 **훈련 곡선**: 공부(train)와 쪽지시험(val) 점수가 함께 오르는지 보기 (test 는 Step 6 까지 봉인 🔒)
- 🎲 **데이터 증강**: 사진을 돌리고·뒤집어 더 많이 공부한 효과 — 단, **색이 곧 정답**이라 색조는 안 바꾸고 밝기·대비도 아주 조금만(±5%)

## ✅ 시작 전 체크

1. **T4 GPU 켜기 (맨 먼저!)** — 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** → 저장
2. **준비할 파일 1개** — Step 3 Part 4(선택)에서 만든 **`dataset.zip`** (내 폴더 `C:\BrainAISorter-…\dataset.zip`)

## 🗺️ 이 단계 전체 그림

| 순서 | 무엇을 하나 | 주의 |
|------|------------|------|
| **Part 1** | 라이브러리 설치 | 런타임이 자동 재시작돼요 — 정상! |
| **Part 2** | 버전·GPU 확인 | 재시작 후 **여기서부터** |
| **Part 3** | `dataset.zip` 올리기 | |
| **Part 4** | 훈련 설정 | 처음엔 그대로 |
| **Part 5** | 데이터 준비 · 증강 | |
| **Part 6** | 모델 만들기 | MobileNetV3 + 내 분류기 |
| **Part 7** | 훈련 ⭐ | 약 5~10분 |
| **Part 8** | 결과 확인 | 정확도 · 혼동행렬 |
| **Part 9** | 모델 내려받기 | **세션이 끊기기 전에 꼭!** |

In [ ]:
# 🛑 Colab 이 아니면 멈춥니다 (PC 에서 실수로 실행하는 것을 막아요)
import sys
if "google.colab" not in sys.modules:
    print("❌ 이 노트북은 Google Colab 전용이에요. 사이트의 'Colab 에서 열기' 버튼을 눌러 주세요.")
    import os
    os.kill(os.getpid(), 9)
print("✅ Colab 에서 실행 중")

## Part 1. 라이브러리 설치

PC 와 **같은 버전(TensorFlow 2.19 · Keras 3.11)** 으로 맞춰야 PC 에서 모델을 열 수 있어요.
실행하면 설치 후 **런타임이 자동으로 재시작**돼요.

> 💡 "세션이 다운되었습니다" 메시지 → **정상**이에요. 닫고 **Part 2 부터** 이어서 실행하세요.

In [ ]:
!pip install -q tensorflow==2.19.0 keras==3.11.2
import IPython
IPython.Application.instance().kernel.do_shutdown(True)   # 런타임 재시작

## Part 2. 버전 · GPU 확인

> 🔄 **재시작 후 여기서부터** 실행하세요.

In [ ]:
import tensorflow as tf
import keras

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
gpu = len(tf.config.list_physical_devices("GPU")) > 0
print(f"GPU        : {'✅ 사용 중' if gpu else '❌ 없음 — 런타임 유형을 T4 GPU 로 바꾸세요'}")
assert tf.__version__.startswith("2.19") and keras.__version__.startswith("3.11"), \
    "버전이 달라요. Part 1 을 다시 실행하세요."

## Part 3. `dataset.zip` 올리기

아래 셀을 실행하고 **파일 선택**에서 Step 3 의 `dataset.zip` 을 고르세요.
(왼쪽 📁 패널에 직접 끌어다 놓아도 돼요 — 그럴 땐 이 셀을 건너뛰세요)

In [ ]:
from google.colab import files
uploaded = files.upload()
print("✅ 올린 파일:", list(uploaded))

In [ ]:
import json, os, zipfile

if not os.path.isdir("dataset"):
    with zipfile.ZipFile("dataset.zip") as archive:
        archive.extractall(".")

info = json.load(open("dataset/info.json", encoding="utf-8"))
KIT = info["kit"]
CLASS_NAMES = info["classes"]          # 0 none, 1 … = micro:bit 명령 번호 순서
MODEL_NAME = f"sorter_{KIT}"

print(f"키트: {KIT}  |  클래스 {len(CLASS_NAMES)}개: {CLASS_NAMES}")
for name, c in info["counts"].items():
    print(f"  {name:<7} train {c['train']:>4}  val {c['val']:>4}  test {c.get('test', 0):>4} (Step 6 전용)")

## Part 4. 훈련 설정

| 설정 | 기본값 | 뜻 |
|------|--------|-----|
| `EPOCHS` | 30 | 전체 사진을 몇 바퀴 공부할지 (더 안 좋아지면 자동으로 일찍 멈춤) |
| `BATCH_SIZE` | 32 | 한 번에 보는 사진 수 |
| `LEARNING_RATE` | 0.001 | 한 번에 얼마나 크게 고칠지 |
| `WEIGHTS` | `"imagenet"` | ImageNet 으로 미리 학습된 MobileNetV3 의 값(가중치) 사용 (전이학습) |

In [ ]:
IMAGE_SIZE    = 224
EPOCHS        = 30
BATCH_SIZE    = 32
LEARNING_RATE = 0.001
SEED          = 42
WEIGHTS       = "imagenet"     # 전이학습: ImageNet 으로 미리 공부한 가중치

keras.utils.set_random_seed(SEED)
print(f"{MODEL_NAME}: {EPOCHS} epochs · batch {BATCH_SIZE} · lr {LEARNING_RATE}")

## Part 5. 데이터 준비 · 증강

- 정답은 Step 1 에서 저장한 **라벨(`.json`)** 이에요. Step 3 이 라벨을 보고 사진을 `dataset/` 의 색별 칸에 나눠 담아 두었어요 → 여기서는 그 칸별로 사진을 읽기만 해요. 클래스 **순서는 `labels.txt` 와 똑같이** 맞춰요.
- 사진을 **[-1, 1]** 로 바꿔요 → PC(Step 5~7)의 `sorter_model.preprocess()` 와 **똑같은 전처리**
- **증강(훈련 사진만)**: 뒤집기 · 회전은 마음껏, 밝기 · 대비는 **±5% 만 아주 조금**, 노이즈 약간. **색조(hue)는 바꾸지 않아요.** 색이 곧 정답이라, 색이 달라 보이게 하는 변화는 조심해야 해요 — 색조를 바꾸면 빨강이 주황이 되고, 밝기를 많이 낮추면 주황이 갈색처럼 보여요.

In [ ]:
from keras import layers

def load(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        f"dataset/{split}", labels="inferred", label_mode="categorical",
        class_names=CLASS_NAMES, image_size=(IMAGE_SIZE, IMAGE_SIZE),
        interpolation="area", batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED)

train_raw = load("train", shuffle=True)
val_raw = load("val", shuffle=False)

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.5, fill_mode="reflect"),        # 초콜릿은 어느 방향이든 같은 색
    layers.RandomBrightness(0.05, value_range=(0, 255)),    # 아주 조금만 — 많이 바꾸면 주황↔갈색처럼 색이 달라 보여요
    layers.RandomContrast(0.05),
], name="augment")

def scale(images):                                          # [0,255] → [-1,1]  (PC 와 동일)
    return images / 127.5 - 1.0

def add_noise(images):
    return images + tf.random.normal(tf.shape(images), stddev=0.02)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(lambda x, y: (add_noise(scale(augment(x, training=True))), y),
                         num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_raw.map(lambda x, y: (scale(x), y), num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
print("✅ 데이터 준비 완료")

In [ ]:
# 증강이 어떻게 되는지 눈으로 보기 (같은 사진 6가지 버전)
import matplotlib.pyplot as plt
images, labels = next(iter(train_raw))
plt.figure(figsize=(9, 1.8))
for i in range(6):
    plt.subplot(1, 6, i + 1)
    plt.imshow(augment(images[:1], training=True)[0].numpy().clip(0, 255).astype("uint8"))
    plt.axis("off")
plt.suptitle(f"augmentation examples - label: {CLASS_NAMES[int(labels[0].numpy().argmax())]}")
plt.show()

## Part 6. 모델 만들기

```
사진(내 훈련 데이터, 224×224) → 특징 뽑기(MobileNetV3 앞부분, 고정🔒) → 요약하기(평균 풀링) → 판단하기(Dense 64) → 외우기 방지(Dropout) → 색 고르기(softmax)
```
| 칸 | 하는 일 | 숫자로 보면 |
|----|--------|-----------|
| ① 사진 | 내가 모은 **훈련 데이터**(입력). 정답(라벨)과 함께 들어가요 | 224 × 224 × 3(RGB) = 약 15만 개 숫자 |
| ② 특징 뽑기 (MobileNetV3 앞부분 🔒) | 미리 학습된 모델이 모양·반짝임·색 기운 같은 특징을 찾아요. 원래 마지막 판단 부분은 떼고(`include_top=False`) 앞부분만 **고정(얼림)** → `base.trainable = False` | 7 × 7 자리마다 특징 720가지 |
| ③ 요약하기 (평균 풀링) | 같은 특징의 49자리 점수를 평균 → 초콜릿 위치와 상관없게 | **720개** |
| ④ 판단하기 (Dense 64) ✏️ | 뉴런 64개가 720개 숫자를 저마다 다른 비중(가중치)으로 더해요. **훈련 = 이 비중을 고치는 것** | 720 → **64개** |
| ⑤ 외우기 방지 (Dropout 20%) | 훈련할 때만 약 13개를 무작위로 0 → 몇 개에만 기대어 외우지 못하게. 평가·사용 때는 모두 켜요 | 64개 |
| ⑥ 색 고르기 (Dense + softmax) ✏️ | 색마다 점수 → softmax 가 합 100% 확률로. 가장 높은 색이 답, 이 확률이 Step 7 의 '확신도' | 클래스 수만큼 확률 (이진 3개 · 다중 7개) |

✏️ = 내 사진으로 새로 배우는 칸(④·⑥). ②는 고정, ③·⑤는 배울 값이 없는 계산 규칙이에요.

- 앞부분을 고정하니 새로 배우는 숫자가 아주 적어요 → 사진이 적어도 잘 배워요
- `include_preprocessing=False` → 스케일은 Part 5 에서 **딱 한 번만** 했으니까요

In [ ]:
# ── ② 특징 뽑기: 미리 학습된 MobileNetV3 Large 를 가져와요 ───────────────────
base = keras.applications.MobileNetV3Large(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),   # 224 × 224 × 3(RGB) 사진을 받아요
    alpha=0.75,                    # 모델 크기 0.75배 — 가볍고 빠르게
    include_top=False,             # 원래 1000가지 물체를 고르던 '마지막 판단 부분'은 떼어 내요
    weights=WEIGHTS,               # "imagenet" = 100만 장 이상으로 미리 학습된 값을 내려받아 써요
    include_preprocessing=False)   # [-1, 1] 스케일은 앞(Part 3)에서 이미 했어요 → 두 번 하지 않게
base.trainable = False             # 🔒 고정(얼림): 훈련해도 이 부분의 값은 바뀌지 않아요

# ── ① 사진 → ②~⑥ 을 차례로 연결해 '나의 색 분류 모델'을 만들어요 ─────────────
inputs = keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3))                # ① 사진 (내가 모은 데이터)
features = base(inputs, training=False)                          # ② 특징 뽑기 → 7 × 7 자리 × 특징 720가지
pooled = layers.GlobalAveragePooling2D()(features)               # ③ 요약하기 → 특징마다 49자리 평균 → 720개
hidden = layers.Dense(64, activation="relu")(pooled)             # ④ 판단하기 ✏️ → 뉴런 64개 (relu: 음수는 0)
dropped = layers.Dropout(0.2)(hidden)                            # ⑤ 외우기 방지 → 훈련 때만 20% 를 무작위로 0
outputs = layers.Dense(len(CLASS_NAMES), activation="softmax")(dropped)   # ⑥ 색 고르기 ✏️ → 색마다 확률(합 100%)
model = keras.Model(inputs, outputs, name=MODEL_NAME)

# ── 훈련 방법 정하기 ─────────────────────────────────────────────────────
model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),   # 틀린 만큼 ④·⑥ 의 가중치를 조금씩 고치는 방법
              loss="categorical_crossentropy",                   # '얼마나 틀렸나' 점수 — 정답 색 확률이 낮을수록 커요
              metrics=["accuracy"])                              # 사람이 보기 쉬운 점수: 맞힌 비율

# 칸마다 숫자 크기 확인 (그림의 ①~⑥ 과 비교해 보세요)
print(f"① 사진 {tuple(inputs.shape[1:])} → ② 특징 {tuple(features.shape[1:])} → ③ 요약 {tuple(pooled.shape[1:])}"
      f" → ④ 판단 {tuple(hidden.shape[1:])} → ⑥ 색 확률 {tuple(outputs.shape[1:])}")
trainable = sum(int(tf.size(w)) for w in model.trainable_weights)
print(f"전체 파라미터 {model.count_params():,}개 중 새로 공부하는 것(④·⑥) {trainable:,}개")

## Part 7. 훈련 ⭐

> ⏱️ T4 GPU 기준 약 5~10분. `val_accuracy`(쪽지시험 점수)가 오르는지 보세요.
> 더 좋아지지 않으면 **자동으로 일찍 멈추고 가장 좋았던 순간**으로 돌아가요.
>
> 🛑 **조기 종료(EarlyStopping) — 훈련을 언제 멈출지 val 로 판단해요**
> - 한 바퀴가 끝날 때마다 val 로 쪽지시험을 봐요
> - 쪽지시험 점수(`val_loss`)가 **10바퀴 연속 나아지지 않으면** 30바퀴를 다 채우지 않고 멈춰요
> - 멈추면 **가장 좋았던 바퀴의 모델**로 되돌려 저장해요
> - 5바퀴 동안 안 나아지면 공부 속도(learning rate)를 절반으로 줄이는 것도 val 로 정해요

In [ ]:
os.makedirs(f"models/{MODEL_NAME}", exist_ok=True)
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

## Part 8. 결과 확인

- **두 선이 함께 오르면 ✅**, train 만 오르고 val 이 멈추면 **외우기만 한 것(과적합)** — 데이터를 더 모아요
- **혼동행렬**: 줄 = 정답, 칸 = AI 예측. 대각선 밖 숫자 = 헷갈린 횟수
- ⚠️ 여기 점수는 **val(쪽지시험)** 기준이라 실제보다 조금 높게 나올 수 있어요 → **최종 합격 판정은 Step 6 의 test** 로

In [ ]:
import numpy as np

h = history.history
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(h["accuracy"], label="train"); axes[0].plot(h["val_accuracy"], label="val")
axes[0].set_title("accuracy"); axes[0].legend()
axes[1].plot(h["loss"], label="train"); axes[1].plot(h["val_loss"], label="val")
axes[1].set_title("loss"); axes[1].legend()
plt.show()

y_true = np.concatenate([y.numpy().argmax(1) for _, y in val_ds])
y_pred = model.predict(val_ds, verbose=0).argmax(1)
val_accuracy = float((y_true == y_pred).mean())
matrix = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=int)
for t, p in zip(y_true, y_pred):
    matrix[t, p] += 1

print(f"검증 정확도: {val_accuracy:.3f}")
print("(참고용 — val 은 훈련 중 '훈련을 언제 멈출지(조기 종료)' 판단하는 데 썼어요. 진짜 성적은 Step 6 의 test 로 재요)")
print("✅ 0.95 이상 — 좋아요, Step 5 로" if val_accuracy >= 0.95 else
      "🟡 0.85~0.95 — 헷갈리는 색을 더 모으면 좋아요" if val_accuracy >= 0.85 else
      "❌ 0.85 미만 — Step 1 로 돌아가 데이터를 보강하세요")
print("\n혼동행렬 (줄=정답, 칸=예측)")
print("        " + " ".join(f"{n[:6]:>6}" for n in CLASS_NAMES))
for name, row in zip(CLASS_NAMES, matrix):
    print(f"{name[:7]:<8}" + " ".join(f"{v:>6}" for v in row))

## Part 9. 모델 내려받기 💾

> ⚠️ **Colab 은 세션이 끊기면 파일이 모두 사라져요. 훈련 직후 바로 내려받으세요!**

`sorter_…_model.zip` 이 내려받아져요. 이 파일을 **PC 의 코스 폴더**(`C:\BrainAISorter-…\`)에 그대로 넣으면 Step 5 가 알아서 풀어요.

In [ ]:
import datetime, shutil

folder = f"models/{MODEL_NAME}"
model.save(f"{folder}/{MODEL_NAME}.keras")
shutil.copy("dataset/labels.txt", f"{folder}/labels.txt")
json.dump({
    "kit": KIT, "classes": CLASS_NAMES, "val_accuracy": round(val_accuracy, 4),
    "epochs_run": len(h["loss"]), "image_size": IMAGE_SIZE, "weights": WEIGHTS,
    "preprocess": "RGB, resize 224 (area), x/127.5-1", "tensorflow": tf.__version__,
    "keras": keras.__version__, "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
}, open(f"{folder}/training_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

ZIP_NAME = f"{MODEL_NAME}_model.zip"
with zipfile.ZipFile(ZIP_NAME, "w") as archive:
    for file in os.listdir(folder):
        archive.write(f"{folder}/{file}", file)
print(f"✅ {ZIP_NAME} 준비 ({os.path.getsize(ZIP_NAME) / 1024 / 1024:.1f} MB)")

In [ ]:
from google.colab import files
files.download(ZIP_NAME)

In [ ]:
# (선택) Google Drive 에도 백업 — 다운로드가 안 될 때
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/BrainAI/sorter", exist_ok=True)
shutil.copy(ZIP_NAME, "/content/drive/MyDrive/BrainAI/sorter/")
print("✅ 구글 드라이브 → BrainAI → sorter 폴더에 저장")

## 다음 ▶️ Step 5 · Model Optimization (PC)

```
C:\BrainAISorter-…\
├── sorter_multi_model.zip   ← 여기에 넣기 (이진분류기는 sorter_binary_model.zip)
└── Step 5 - Model Optimization.ipynb
```